# Predicting Smartphone Addiction

In [1]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [2]:
train_df = pd.read_csv('data/train.csv')

train_df.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [3]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       691369 non-null  int64  
 1   age                      662440 non-null  float64
 2   daily_screen_time_hours  595515 non-null  float64
 3   social_media_hours       557374 non-null  float64
 4   gaming_hours             564548 non-null  float64
 5   work_study_hours         639851 non-null  float64
 6   sleep_hours              646889 non-null  float64
 7   notifications_per_day    623785 non-null  float64
 8   app_opens_per_day        610659 non-null  float64
 9   weekend_screen_time      579306 non-null  float64
 10  gender                   662335 non-null  str    
 11  stress_level             636221 non-null  str    
 12  academic_work_impact     647145 non-null  str    
 13  addicted_label           691369 non-null  int64  
dtypes: float64(9), 

In [4]:
CAT = ['gender', 'stress_level', 'academic_work_impact']

for col in CAT:
    print(col, train_df[col].unique())

gender <StringArray>
['Male', 'Female', 'Other', nan]
Length: 4, dtype: str
stress_level <StringArray>
['Medium', 'Low', 'High', nan]
Length: 4, dtype: str
academic_work_impact <StringArray>
['No', 'Yes', nan]
Length: 3, dtype: str


In [5]:
DROP = ['id']

train_df = train_df.drop(columns=DROP)

In [6]:
TARGET = 'addicted_label'

cut = int(len(train_df) * 0.8)

X = train_df.drop(columns=TARGET)
y = train_df[TARGET]

X_train, X_test = X[:cut], X[cut:]
y_train, y_test = y[:cut], y[cut:]


In [7]:
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), CAT),
    ],
    remainder='passthrough'
)

pipeline = Pipeline(
    steps=[
        ('pre', preprocess),
        ('model', LGBMClassifier(random_state=42))
    ]

)

In [8]:
pipeline.fit(X_train, y_train)

feature_importances = pipeline.named_steps['model'].feature_importances_
feature_names = pipeline.named_steps['pre'].get_feature_names_out()

fi_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances
}).sort_values("importance", ascending=False)

print(fi_df)

[LightGBM] [Info] Number of positive: 392259, number of negative: 160836
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002461 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1966
[LightGBM] [Info] Number of data points in the train set: 553095, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.709207 -> initscore=0.891537
[LightGBM] [Info] Start training from score 0.891537
                               feature  importance
18        remainder__app_opens_per_day         743
17    remainder__notifications_per_day         613
12  remainder__daily_screen_time_hours         389
13       remainder__social_media_hours         358
19      remainder__weekend_screen_time         354
15         remainder__work_study_hours         187
14             remainder__gaming_hours         161
11                      remainder

In [9]:
y_pred = pipeline.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred)

print('ROC AUC: ', auc)

ROC AUC:  0.9546096355861663


In [10]:
test_df = pd.read_csv('data/test.csv')

test_ids = test_df['id']
test_df = test_df.drop(columns=DROP)

test_pred = pipeline.predict_proba(test_df)[:, 1]

submission = pd.DataFrame({
    'id': test_ids,
    'addicted_label': test_pred
})

assert len(submission) == len(test_df), "Row count mismatch"
assert not submission.isna().any().any(), "Missing values in submission"

# Save
submission.to_csv('submission.csv', index=False)